# Train Decision Segmenter — CausaGanha

Fine-tunes **`openai/privacy-filter`** (token classifier, Apache 2.0) to **identify and segment** the three structural parts of a Brazilian judicial decision.

| Label | Section | Description |
|---|---|---|
| `RELATORIO` | Relatório | Case history and facts summary |
| `FUNDAMENTACAO` | Fundamentação | Legal reasoning |
| `DISPOSITIVO` | Dispositivo | Operative ruling (the actual decision) |

**Why `openai/privacy-filter` as base?**
- Already a token classifier — we just replace its 33-class PII head with a 3-class section head.
- 128K-token context window (handles complete judicial decisions in one pass).
- Token-level labels give **exact character boundaries** between sections, not just paragraph labels.
- Apache 2.0 license; open weights.

**Architecture note**: uses banded attention (128-token window per token). That's fine here — section boundaries are marked by local phrases (`ante o exposto`, `RELATÓRIO`, etc.), not global document context.

> **Runtime**: GPU (T4). Enable via Runtime → Change runtime type.

## 1. Setup — clone repo & install deps

In [ ]:
REPO_URL  = "https://github.com/franklinbaldo/causaganha.git"
BRANCH    = "feat/embedder-smart-truncate-and-privacy-dataset-v2"
REPO_DIR  = "/content/causaganha"


In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version


In [ ]:
!uv pip install --system -e ".[embeddings]" transformers accelerate datasets seqeval


## 2. Download data from Internet Archive

In [ ]:
import os, urllib.request

PARQUET_DIR = f"{REPO_DIR}/data/test_parquets"
os.makedirs(PARQUET_DIR, exist_ok=True)

FILES = {
    "textos.parquet": "https://archive.org/download/causaganha-test-parquets/textos.parquet",
}

for fname, url in FILES.items():
    dest = os.path.join(PARQUET_DIR, fname)
    if os.path.exists(dest):
        print(f"  Already exists: {fname}")
        continue
    print(f"  Downloading {fname} ...")
    urllib.request.urlretrieve(url, dest)
    size = os.path.getsize(dest)
    print(f"  OK {fname} ({size:,} bytes)")

print('All files ready.')


## 3. Prepare labeled dataset

Uses heuristic markers (`ante o exposto`, `fundamentação`, etc.) to assign **character-level section spans** to each document. These are silver labels — good enough to bootstrap a model that will outperform the heuristic on unseen text, especially cases without explicit markers.

Each document becomes one training example with char-level spans. Tokenization (next cell) aligns these spans to token-level labels.

In [ ]:
import sys, re, random
import numpy as np
import ibis
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")

_DISPOSITIVO_RE = re.compile(
    r'(?:ante\s+o\s+exposto|posto\s+isso|isso\s+posto|'
    r'diante\s+do\s+exposto|pelo\s+exposto|em\s+face\s+do\s+exposto|'
    r'por\s+tais\s+fundamentos|nestes\s+termos|em\s+conclus[\u00e3a]o|'
    r'pelo\s+que\s+exposto|em\s+vista\s+do\s+exposto)',
    re.IGNORECASE,
)
_FUNDAMENTACAO_RE = re.compile(
    r'(?:fundament[ao](?:\u00e7\u00e3o)?|m[\u00e9e]rito|an[\u00e1a]lise\s+do\s+pedido|'
    r'da\s+an[\u00e1a]lise|do\s+m[\u00e9e]rito|'
    r'fundamenta[\u00e7c][\u00e3a]o\s+(?:jur[\u00edi]dica|do\s+ju[\u00edi]zo))',
    re.IGNORECASE,
)

def segment_document(text):
    """Return char-level span dict or None if dispositivo not found."""
    disp_m = _DISPOSITIVO_RE.search(text)
    if not disp_m:
        return None
    disp_start = disp_m.start()
    pre = text[:disp_start]
    fund_m = _FUNDAMENTACAO_RE.search(pre)
    fund_start = fund_m.start() if fund_m else len(pre) // 2
    spans = {}
    if fund_start > 0:
        spans['relatorio'] = [[0, fund_start]]
    if disp_start > fund_start:
        spans['fundamentacao'] = [[fund_start, disp_start]]
    spans['dispositivo'] = [[disp_start, len(text)]]
    return spans

t = ibis.read_parquet(Path(PARQUET_DIR) / 'textos.parquet')
df = t.filter(t.texto.notnull()).execute()
print(f'Loaded {len(df):,} documents')

records, skipped = [], 0
for _, row in df.iterrows():
    spans = segment_document(row['texto'])
    if spans is None:
        skipped += 1
        continue
    records.append({'text': row['texto'], 'spans': spans})

print(f'Documents with spans: {len(records):,} (skipped {skipped} without dispositivo)')


## 4. Build HuggingFace Dataset

In [ ]:
from datasets import Dataset

random.seed(42)
random.shuffle(records)
n = len(records)
train_end = int(n * 0.8)
val_end   = train_end + int(n * 0.1)

raw_train = Dataset.from_list(records[:train_end])
raw_val   = Dataset.from_list(records[train_end:val_end])
raw_test  = Dataset.from_list(records[val_end:])

print(f'Train: {len(raw_train):,}  Val: {len(raw_val):,}  Test: {len(raw_test):,}')


## 5. Tokenize + align labels to tokens

`openai/privacy-filter` is already a token classifier — we load it with `num_labels=3` replacing its 33-class PII head with a fresh 3-class section head. Token labels are aligned from the character spans using `return_offsets_mapping=True`. Special tokens (CLS/SEP) get label `-100` (ignored in loss).

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "openai/privacy-filter"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ID2LABEL = {0: 'RELATORIO', 1: 'FUNDAMENTACAO', 2: 'DISPOSITIVO'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

def tokenize_and_label(example):
    text  = example['text']
    spans = example['spans']

    # Build char-level label array (default: RELATORIO=0)
    char_labels = np.zeros(len(text), dtype=np.int32)
    for section, span_list in spans.items():
        lid = LABEL2ID.get(section.upper(), 0)
        for start, end in span_list:
            char_labels[start:min(end, len(text))] = lid

    enc = tokenizer(
        text,
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
    )
    offsets = enc.pop('offset_mapping')

    token_labels = []
    for start, end in offsets:
        if start == end:   # special token
            token_labels.append(-100)
        else:
            token_labels.append(int(char_labels[start]))

    enc['labels'] = token_labels
    return enc

train_ds = raw_train.map(tokenize_and_label, remove_columns=['text', 'spans'])
val_ds   = raw_val.map(tokenize_and_label,   remove_columns=['text', 'spans'])
test_ds  = raw_test.map(tokenize_and_label,  remove_columns=['text', 'spans'])

print('Tokenization done.')
print(f'  Example token count: {len(train_ds[0]["input_ids"])}')
print(f'  Label distribution in first doc:', {ID2LABEL[l]: train_ds[0]["labels"].count(l) for l in range(3)})


## 6. Fine-tune

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np
from seqeval.metrics import classification_report as seq_report

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # replaces 33-class PII head with 3-class section head
)

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)
    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seq, pred_seq = [], []
        for p_id, l_id in zip(pred_row, label_row):
            if l_id == -100:
                continue
            true_seq.append(ID2LABEL[l_id])
            pred_seq.append(ID2LABEL[p_id])
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    report = seq_report(true_seqs, pred_seqs, output_dict=True, zero_division=0)
    return {
        'macro_f1':           report.get('macro avg', {}).get('f1-score', 0),
        'f1_relatorio':       report.get('RELATORIO', {}).get('f1-score', 0),
        'f1_fundamentacao':   report.get('FUNDAMENTACAO', {}).get('f1-score', 0),
        'f1_dispositivo':     report.get('DISPOSITIVO', {}).get('f1-score', 0),
    }

total_steps = (len(train_ds) // 16) * 3
warmup_steps = max(50, total_steps // 10)

training_args = TrainingArguments(
    output_dir="/content/decision_segmenter",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=True,
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()


## 7. Evaluate on test set

In [ ]:
preds_out = trainer.predict(test_ds)
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids

true_seqs, pred_seqs = [], []
for pred_row, label_row in zip(preds, labels):
    t, p = [], []
    for p_id, l_id in zip(pred_row, label_row):
        if l_id == -100:
            continue
        t.append(ID2LABEL[l_id])
        p.append(ID2LABEL[p_id])
    true_seqs.append(t)
    pred_seqs.append(p)

print(seq_report(true_seqs, pred_seqs, zero_division=0))


## 8. Save & download model

In [ ]:
import shutil
from google.colab import files

MODEL_OUT = "/content/decision_segmenter_best"
trainer.save_model(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print(f"Model saved to {MODEL_OUT}")

shutil.make_archive('/content/decision_segmenter', 'zip', MODEL_OUT)
files.download('/content/decision_segmenter.zip')
print('Downloaded decision_segmenter.zip')
